In [ ]:
%pip install agent-framework-openai -U
%pip install RateMyProfessorAPI

In [ ]:
import os
from dotenv import load_dotenv
from agent_framework import Agent, tool, AgentRunInputs, ResponseStream
from agent_framework.openai import OpenAIChatCompletionClient
from typing import Annotated
import requests
import re
import pandas as pd

load_dotenv()

In [ ]:
openai_chat_client = OpenAIChatCompletionClient(
    base_url=os.environ.get("GITHUB_ENDPOINT"),
    api_key=os.environ.get("GITHUB_TOKEN"),
    model=os.environ.get("GITHUB_MODEL_ID")
)

In [ ]:
# Time formatting
def _fmt_time(t):
    if not t or not t.isdigit() or len(t) < 4:
        return t or 'N/A'
    hour, minute = int(t[:2]), t[2:4]
    if 8 <= hour <= 11:
        return f"{hour}:{minute} AM"
    elif hour == 12:
        return f"12:{minute} PM"
    elif 1 <= hour <= 7:
        return f"{hour}:{minute} PM"
    return f"{hour}:{minute}"

# Time validation
def _is_valid_start(t):
    if not t or not t.isdigit() or len(t) < 4:
        return False
    return t >= '0100'

# Time preference filter
def _matches_time_pref(t, pref):
    """morning: 0800–1159 | afternoon: 1200–1659 | evening: 1700+"""
    if pref == "any" or not t or not t.isdigit() or len(t) < 4:
        return True
    if pref == "morning":
        return '0800' <= t < '1200'
    if pref == "afternoon":
        return '1200' <= t < '1700'
    if pref == "evening":
        return t >= '1700'
    return True

# Day code mapping
_DAY_MAP = {"M": "Mon", "T": "Tue", "W": "Wed", "TH": "Thu", "F": "Fri", "S": "Sat"}

# Subject code mapping
SUBJECT_MAP = {
    "cs": "198", "computer science": "198",
    "data science": "954", "ds": "954",
    "math": "640", "mathematics": "640",
    "physics": "750",
    "biology": "119", "bio": "119",
    "chemistry": "160", "chem": "160",
    "economics": "220", "econ": "220",
    "psychology": "830", "psych": "830",
    "statistics": "960", "stats": "960",
    "electrical engineering": "332", "ee": "332",
    "mechanical engineering": "650", "me": "650",
    "civil engineering": "250",
    "finance": "553",
    "management": "620",
    "accounting": "533",
    "history": "510",
    "english": "350",
    "linguistics": "615",
    "philosophy": "730",
    "sociology": "920",
    "political science": "790",
}

_last_instructors: list[str] = []

#fetche courses from the Rutgers SOC API
@tool
def get_rutgers_courses(subject: str = "198", time_preference: str = "any"):
    """
    Fetches courses from Rutgers for the given subject and time preference.
    subject: major name (e.g. 'Computer Science', 'Math') or a raw subject code (e.g. '198').
    time_preference: 'morning' (before noon), 'afternoon' (noon–5 PM), 'evening' (5 PM+), or 'any'.
    """
    global _last_instructors
    subject_code = SUBJECT_MAP.get(subject.lower().strip(), subject.strip())
    pref = time_preference.lower().strip()

    url = f"https://classes.rutgers.edu/soc/api/courses.json?year=2026&term=9&campus=NB&subject={subject_code}"

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        all_courses = response.json()

        if not all_courses:
            return "No courses found for this subject/term."

        matching_classes = []
        for course in all_courses:
            course_str = course.get('courseString', '')
            if f':{subject_code}:' not in course_str:
                continue
            sections = course.get('sections', [])
            if not sections:
                continue
            section = sections[0]
            meetings = section.get('meetingTimes', [])
            meeting = meetings[0] if meetings else {}
            start_time = meeting.get('startTime', '')
            if not _is_valid_start(start_time):
                continue
            if not _matches_time_pref(start_time, pref):
                continue
            end_time = meeting.get('endTime', '')
            matching_classes.append({
                "title": course.get('title', 'Unknown').title(),
                "course_id": course_str,
                "instructor": section.get('instructorsText', 'Staff').title(),
                "time_display": f"{_fmt_time(start_time)} – {_fmt_time(end_time)}",
                "day": _DAY_MAP.get(meeting.get('meetingDay', 'N/A'), meeting.get('meetingDay', 'N/A'))
            })

        results = matching_classes[:20]

        if not results:
            pref_label = f"{pref} " if pref != "any" else ""
            return f"No {pref_label}courses found for subject code {subject_code}."

        seen = set()
        _last_instructors = []
        for c in results:
            name = c['instructor']
            if name.lower() != 'staff' and name not in seen:
                seen.add(name)
                _last_instructors.append(name)

        rows = "\n".join(
            f"| {i+1} | {c['title']} | `{c['course_id']}` | {c['instructor']} | {c['time_display']} | {c['day']} |"
            for i, c in enumerate(results)
        )
        table = (
            "| # | Course Title | Course ID | Instructor | Time | Day |\n"
            "|:-:|-------------|:---------:|-----------|:----:|:---:|\n"
            + rows
        )
        pref_label = f"{pref.title()} " if pref != "any" else ""
        meta = f"> Campus: New Brunswick | Subject Code: {subject_code} | Filter: {pref_label}classes\n\n"

        with open(OUTPUT_FILE, "a") as f:
            f.write(f"\n## {pref_label}Classes\n\n" + meta + table + "\n\n---\n")

        return results
    except Exception as e:
        return f"API Error: {str(e)}"

#returns instructor names collected in course search
@tool
def get_instructor_names() -> str:
    """Returns the comma-separated list of all instructors from the most recent get_rutgers_courses call."""
    if not _last_instructors:
        return "No instructors found — run get_rutgers_courses first."
    return ", ".join(_last_instructors)

In [ ]:
RUTGERS_SCHOOL_ID = "4996"
RMP_HEADERS = {'User-Agent': 'Mozilla/5.0'}

# Difficulty bar
def _diff_emoji_bar(d):
    d = d or 0
    filled = round(d * 2)
    if d >= 4.0:
        block = "🟥"
    elif d >= 3.0:
        block = "🟧"
    elif d >= 2.0:
        block = "🟨"
    else:
        block = "🟩"
    return block * filled + "⬜" * (10 - filled)

# Star rating
def _star_bar(r):
    r = r or 0
    full = int(r)
    empty = 5 - full
    return "⭐" * full + "☆" * empty

#fetches professor difficulty + rating from RateMyProfessor
@tool
def get_professor_difficulty(professor_names: str):
    """Fetches RMP difficulty and rating for Rutgers professors (comma-separated names)."""
    names = [n.strip() for n in professor_names.split(",") if n.strip() and n.strip().lower() != 'staff']
    if not names:
        return "No specific instructors found."

    results = []
    for name in names:
        try:
            search_url = f"https://www.ratemyprofessors.com/search/professors/{RUTGERS_SCHOOL_ID}?q={name}"
            page = requests.get(search_url, headers=RMP_HEADERS, timeout=10)
            prof_ids = re.findall(r'"legacyId":(\d+)', page.text)

            if not prof_ids:
                results.append(f"### ❓ {name}\n\n*Not found on RateMyProfessors.*\n")
                continue

            prof_url = f"https://www.ratemyprofessors.com/professor/{prof_ids[0]}"
            p = requests.get(prof_url, headers=RMP_HEADERS, timeout=10)

            first_names = re.findall(r'"firstName":"(.*?)"', p.text)
            last_names = re.findall(r'"lastName":"(.*?)"', p.text)
            ratings = re.findall(r'"avgRating":([\d.]+)', p.text)
            difficulties = re.findall(r'"avgDifficulty":([\d.]+)', p.text)

            full_name = f"{first_names[0]} {last_names[0]}" if first_names and last_names else name
            rating = float(ratings[0]) if ratings else None
            difficulty = float(difficulties[0]) if difficulties else None

            if rating is None and difficulty is None:
                results.append(f"### ❓ {name}\n\n*Not found on RateMyProfessors.*\n")
                continue

            bar = _diff_emoji_bar(difficulty)
            stars = _star_bar(rating)

            if difficulty and difficulty >= 4.0:
                diff_label, diff_color = "🔥 **Difficulty**", "#e74c3c"
            elif difficulty and difficulty >= 3.0:
                diff_label, diff_color = "🔥 **Difficulty**", "#e67e22"
            else:
                diff_label, diff_color = "🌿 **Difficulty**", "#27ae60"

            results.append(
                f"### 🧑‍💻 {full_name}\n\n"
                f"| Metric | Score | Visual |\n"
                f"|--------|:-----:|--------|\n"
                f'| <span style="color:{diff_color}">{diff_label}</span> | **{difficulty} / 5.0** | {bar} |\n'
                f'| <span style="color:#2ecc71">**⭐ Rating**</span> | **{rating} / 5.0** | {stars} |\n'
            )
        except Exception:
            results.append(f"### ❓ {name}\n\n*Not found on RateMyProfessors.*\n")

    output = "\n<br>\n\n---\n\n".join(results) if results else "No RMP data found."
    footer = '\n\n> 💡 **Tip:** Sort by ⏰ time to find gaps, and check ⭐ ratings before enrolling!\n>\n> 📊 *Data sourced from [Rutgers SOC API](https://classes.rutgers.edu) & [RateMyProfessors](https://www.ratemyprofessors.com)*\n'

    with open(OUTPUT_FILE, "a") as f:
        f.write("\n## 👩‍🏫 Professor Difficulty & Ratings\n\n<br>\n\n" + output + footer)

    return output

In [ ]:
OUTPUT_FILE = os.path.join(os.getcwd(), "course_recommendations.md")

#names a content block to the output file
@tool
def write_to_doc(content: str, section: str = ""):
    """Appends content to the output markdown doc. Pass a 'section' label like 'Courses' or 'Professor Difficulty'."""
    with open(OUTPUT_FILE, "a") as f:
        if section:
            f.write(f"\n## {section}\n\n")
        f.write(content + "\n")
    return f"Written to {OUTPUT_FILE}"

@tool
def transfer_to_difficulty_expert():
    """Transfer to the DifficultyExpert when the user asks about workload, stress, or professor difficulty."""
    return difficulty_expert

In [ ]:
# Agent system prompts
DIFFICULTY_INSTRUCTIONS = """
You are the DifficultyExpert.
1. Call 'get_instructor_names' to get the exact list of all instructors from the course search.
2. Pass that full comma-separated list directly to 'get_professor_difficulty'. Do not drop any names.
3. Present the returned difficulty scores and ratings clearly, and add a one-sentence stress level summary.
"""
COORDINATOR_INSTRUCTIONS = """
You are the Rutgers Schedule Coordinator.
The user will tell you their major and time preference (morning, afternoon, or evening).
1. Extract the major and time preference from the user's message.
2. Call 'get_rutgers_courses' with both the subject (major name) and time_preference arguments.
3. Display ALL returned courses as a Markdown table: Course Title, Course ID, Instructor, Time, Day.
4. IMMEDIATELY call 'transfer_to_difficulty_expert'. Do not wait for the user.
"""

In [ ]:
# DifficultyExpert agent
difficulty_expert = Agent(
    name="DifficultyExpert",
    client=openai_chat_client,
    instructions=DIFFICULTY_INSTRUCTIONS,
    tools=[get_instructor_names, get_professor_difficulty, write_to_doc]
)

In [ ]:
# ScheduleCoordinator agent
coordinator_agent = Agent(
    name="ScheduleCoordinator",
    client=openai_chat_client,
    instructions=COORDINATOR_INSTRUCTIONS,
    tools=[get_rutgers_courses, transfer_to_difficulty_expert, write_to_doc]
)

In [ ]:
#Show supported majors and prompt the user for their major and time preference
supported = ", ".join(sorted({v.title() for v in [
    "Computer Science", "Data Science", "Math", "Physics", "Biology",
    "Chemistry", "Economics", "Psychology", "Statistics",
    "Electrical Engineering", "Mechanical Engineering", "Civil Engineering",
    "Finance", "Management", "Accounting", "History", "English",
    "Linguistics", "Philosophy", "Sociology", "Political Science"
]}))
print(f"Supported majors: {supported}\n")

user_major = input("What is your major? ").strip()
print()
print("Time preference: morning (before noon) | afternoon (noon–5 PM) | evening (5 PM+) | any")
user_time = input("What time of day do you prefer? ").strip().lower()

if user_time not in ("morning", "afternoon", "evening", "any"):
    user_time = "any"
    print("Unrecognized preference — showing all available times.\n")

with open(OUTPUT_FILE, "w") as f:
    f.write(f"# Rutgers Course Recommendations\n\nMajor: {user_major} | Time Preference: {user_time.title()}\n\n---\n")

query = (
    f"I'm a {user_major} major and I want {user_time} classes. "
    f"What courses are available and how is the workload for each professor?"
)

print(f"\nSearching for {user_time} {user_major} courses...\n")

thread = coordinator_agent.create_session()
await coordinator_agent.run(query, session=thread)
await difficulty_expert.run(
    f"Get professor difficulty ratings for the instructors of the {user_time} {user_major} courses found in this session.",
    session=thread
)